# Machine Learning Training | Solution Notebook

This notebook contains worked solutions for all labs in Module 2. Use it after completing the live labs to compare your approach, check your metrics, and review the reasoning behind each decision.

**Important:** There is no single correct answer for most tasks. These solutions represent strong answer shapes. Your approach may differ and still be valid if it is well-justified.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_curve, roc_curve, auc
)

DATA_DIR = Path("../data")
pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (8, 5)

tickets = pd.read_csv(DATA_DIR / "service_tickets_ml.csv")
segments_df = pd.read_csv(DATA_DIR / "customer_segments.csv")
governance = pd.read_csv(DATA_DIR / "model_governance_scenarios.csv")

print(f"Tickets: {tickets.shape}")
print(f"Segments: {segments_df.shape}")
print(f"Governance: {governance.shape}")

## Lab 1 Solution | Problem Framing

**Business decision:** Should an operations team member prioritise a ticket for early intervention to prevent SLA breach?

**Target variable:** Binary indicator of whether the ticket breaches SLA within 48 hours of creation.

**Prediction unit:** Individual service ticket, assessed at the time of creation or initial triage.

**Candidate features (include):**
- Ticket priority level (categorical): higher priority tickets may have different breach patterns.
- Customer segment or tier: high-value customers may receive faster service, affecting breach rates.
- Ticket category or type: some issue types are inherently harder to resolve quickly.
- Time of day / day of week at creation: staffing levels affect resolution speed.
- Historical ticket volume for the customer: repeat contacts may signal complexity.

**Features to exclude:**
- Resolution time: directly determines the target (leakage).
- Agent who resolved the ticket: not available at prediction time.

**Key assumptions:**
1. Historical SLA breach labels are accurate and consistently applied.
2. The operational process has not changed significantly since the training data was collected.
3. The feature set available at prediction time will remain available in production.

In [ ]:
# Lab 2 Solution | Baseline Classification

print("Available columns:", list(tickets.columns))
print()

# Identify numeric columns suitable for a quick baseline
numeric_cols = tickets.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numeric columns: {numeric_cols}")

# Adapt target and features to whatever columns exist in the dataset
# The code below uses a pattern that works with typical service ticket data.
# Participants should adjust column names to match the actual dataset.

# Example baseline pattern:
# target_col = "sla_breach"  # adjust to actual column name
# feature_cols = [c for c in numeric_cols if c != target_col]
# X = tickets[feature_cols].fillna(0)
# y = tickets[target_col]

# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.3, random_state=42, stratify=y
# )

# model = LogisticRegression(max_iter=1000)
# model.fit(X_train, y_train)
# y_pred = model.predict(X_test)

# print(f"Accuracy:  {accuracy_score(y_test, y_pred):.3f}")
# print(f"Precision: {precision_score(y_test, y_pred):.3f}")
# print(f"Recall:    {recall_score(y_test, y_pred):.3f}")
# print(f"F1:        {f1_score(y_test, y_pred):.3f}")

# Interpretation:
# Recall is the priority metric here. A missed SLA breach (false negative) means
# a ticket that needed early intervention received none. The cost of a false
# positive (unnecessary prioritisation) is lower: an extra review that turns out
# to be unnecessary.

print("\nKey insight: recall matters most because missed breaches have higher cost.")

In [ ]:
# Lab 3 Solution | Customer Segmentation

print("Segment data columns:", list(segments_df.columns))
print()

# Select features for segmentation (adapt to actual column names)
# seg_features = ["annual_value", "transaction_frequency", "product_count"]
# X_seg = segments_df[seg_features].fillna(0)

# scaler = StandardScaler()
# X_scaled = scaler.fit_transform(X_seg)

# km = KMeans(n_clusters=4, random_state=42, n_init=10)
# segments_df["cluster"] = km.fit_predict(X_scaled)

# summary = segments_df.groupby("cluster")[seg_features].agg(["mean", "count"])
# print(summary)

# Segment naming example:
# Cluster 0: "High-Value Active" -- high annual value, high transaction frequency
# Cluster 1: "Dormant Holdings" -- low transactions, moderate product count
# Cluster 2: "New Growth" -- low current value, increasing activity
# Cluster 3: "Steady Core" -- moderate across all dimensions

# Action recommendations:
# High-Value Active: retain with premium service and proactive relationship management
# Dormant Holdings: re-engage with targeted product offers
# New Growth: nurture with onboarding support and cross-sell
# Steady Core: maintain with standard service, monitor for churn signals

print("Key insight: segments must have differentiated actions to be useful.")

In [ ]:
# Lab 4 Solution | Model Evaluation Deep-Dive

# After rebuilding the baseline from Lab 2:
# y_proba = model.predict_proba(X_test)[:, 1]

# Precision-recall curve
# prec, rec, thresholds = precision_recall_curve(y_test, y_proba)
# plt.plot(rec, prec)
# plt.xlabel("Recall")
# plt.ylabel("Precision")
# plt.title("Precision-Recall Curve -- SLA Breach Baseline")
# plt.axhline(y=0.7, color="red", linestyle="--", label="Precision floor = 0.70")
# plt.legend()
# plt.show()

# Error analysis pattern:
# errors = X_test.copy()
# errors["actual"] = y_test.values
# errors["predicted"] = y_pred
# errors["error_type"] = np.where(
#     (errors["predicted"] == 1) & (errors["actual"] == 0), "false_positive",
#     np.where((errors["predicted"] == 0) & (errors["actual"] == 1), "false_negative", "correct")
# )
# print(errors["error_type"].value_counts())

# Calibration note:
# If the model assigns probability 0.7 to cases that breach only 55% of the time,
# the model is overconfident. This matters because operations teams may use the
# probability to decide how urgently to act.

print("Key insight: headline metrics hide important failure patterns.")
print("Always categorise errors and check calibration before recommending deployment.")

In [ ]:
# Lab 5 Solution | Model Comparison

# After training both models on the same split:
# model_lr = LogisticRegression(max_iter=1000).fit(X_train, y_train)
# model_rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)

comparison = pd.DataFrame({
    "Criterion": ["Accuracy", "Precision", "Recall", "F1",
                  "Interpretability", "Governance Burden", "Retraining Complexity"],
    "Logistic Regression": ["~0.78", "~0.72", "~0.68", "~0.70",
                            "High (coefficients)", "Low", "Low"],
    "Random Forest": ["~0.82", "~0.76", "~0.74", "~0.75",
                      "Medium (feature importance)", "Medium", "Medium"]
})
print(comparison.to_string(index=False))

print("\nRecommendation: Random Forest for pilot, with caveats.")
print("Caveats:")
print("1. If interpretability is the top priority (regulatory requirement), use Logistic Regression.")
print("2. If the performance gap narrows with more feature engineering, the simpler model is preferable.")
print("3. Monitor feature importance stability across retraining cycles.")

In [ ]:
# Lab 6 Solution | Monitoring Design

monitoring_plan = pd.DataFrame({
    "Metric": ["Weekly Precision", "Weekly Recall", "Prediction Volume",
               "Feature Drift (PSI)", "Confidence Distribution"],
    "Frequency": ["Weekly", "Weekly", "Daily", "Weekly", "Weekly"],
    "Threshold": ["< 0.70", "< 0.65", "> 2x or < 0.5x baseline",
                  "> 0.15", "Mean shift > 0.10"],
    "Response": ["Investigate; escalate if sustained 2+ weeks",
                 "Review false negatives; check for data quality issues",
                 "Check data pipeline; alert operations",
                 "Flag for retraining review",
                 "Check for population shift; review recent tickets"],
    "Owner": ["Model Owner", "Model Owner", "Data Engineering",
              "Model Risk", "Model Owner"]
})
print(monitoring_plan.to_string(index=False))

print("\nEscalation procedure:")
print("1. Model Owner detects threshold breach in weekly review.")
print("2. Model Owner investigates root cause within 3 business days.")
print("3. If cause is data quality: escalate to Data Engineering.")
print("4. If cause is model degradation: escalate to Model Risk for retraining decision.")
print("5. If sustained degradation (3+ weeks): recommend rollback to manual process.")

In [ ]:
# Lab 7 Solution | Deployment Readiness

readiness = pd.DataFrame({
    "Dimension": ["Technical", "Technical", "Technical",
                  "Operational", "Operational",
                  "Governance", "Governance"],
    "Check": ["Model performance meets threshold on held-out data",
              "Data pipeline tested end-to-end",
              "Monitoring dashboard live and tested",
              "Operations team trained on new workflow",
              "Fallback process documented and rehearsed",
              "Model documentation complete and approved",
              "Model risk sign-off obtained"],
    "Status": ["Ready", "Ready", "In Progress",
               "In Progress", "Not Started",
               "Ready", "Pending"]
})
print(readiness.to_string(index=False))

print("\nTop deployment risks:")
print("1. Operations team not trained (likelihood: medium, impact: high)")
print("2. Monitoring dashboard not ready at launch (likelihood: medium, impact: medium)")
print("3. Data pipeline latency causes stale predictions (likelihood: low, impact: high)")

print("\nRecommendation: Defer pilot by 2 weeks.")
print("Conditions: deploy when monitoring dashboard is live, ops team is trained,")
print("and fallback process is documented and rehearsed.")

## Capstone Solution Shape | AJB Model Recommendation

A strong capstone recommendation will include:

**1. Business Decision:** Early identification of service tickets likely to breach SLA, enabling proactive operations intervention.

**2. Model Recommendation:** Random Forest classifier, selected over logistic regression for its stronger recall performance. Logistic regression remains the fallback if regulatory requirements demand full coefficient-level interpretability.

**3. Key Metric:** Recall at 0.70+ with precision floor of 0.65. The trade-off accepted is a moderate false positive rate (some tickets flagged unnecessarily) in exchange for catching more genuine breach risks.

**4. Governance Requirements:**
- Model documentation approved by Model Risk.
- Monitoring dashboard live before pilot begins.
- Quarterly model review scheduled.
- Rollback procedure documented and rehearsed.

**5. Monitoring Plan:** Weekly precision, recall, and drift metrics with named owners and specific thresholds. Escalation procedure with defined response timelines.

**6. Risk Summary:**
- Data drift from changing ticket patterns (mitigated by weekly drift monitoring).
- Operations team resistance to new workflow (mitigated by training and gradual rollout).
- Model overconfidence on edge cases (mitigated by calibration checks and human review for borderline predictions).

**7. Next Step:** Pilot deployment to one regional operations centre for 8 weeks, with weekly review and defined success criteria (recall > 0.70, precision > 0.65, operations team satisfaction survey > 3.5/5).